# S-D: Reversed During-Inference Ensemble — M4→M1 Confidence-Aware

**Arsitektur:** M4 (conservative screener) → M1 (sensitive specialist reviewer)

**Berbeda dari S-B/S-C (M1→M4):**
- S-B/S-C: Sensitive detector → Conservative reviewer = overcorrection (spec↑ tapi sens↓↓)
- S-D: Conservative screener → Sensitive specialist = additive (catch yang terlewat)

**Inovasi:** Kedua model output confidence score (%) dan key evidence.

**2 Pass:**
1. Pass 1 — M4 first reader: classification + confidence% + key evidence
2. Pass 2 — M1 specialist reviewer: evaluasi M4 + own observation + classification + confidence% + key evidence

**Catatan VRAM:** Setiap pass hanya load 1 model (~3-5 GB 4-bit). Setelah Pass 1 selesai, unload M4 lalu load M1 untuk Pass 2.

## 1 — Setup & Imports

In [ ]:
import sys
import json
import time
import re
import gc
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm

# Add src to path
sys.path.insert(0, str(Path('src').resolve()))
from during_ensemble import (
    build_few_shot_context_v2,
    load_checkpoint, append_jsonl,
)
from evaluate import extract_diagnosis, compute_classification_metrics

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2 — Configuration

In [ ]:
# ════════════════════════════════════════════════════════════════════
# KONFIGURASI — SESUAIKAN PATH INI
# ════════════════════════════════════════════════════════════════════

# Model paths
M4_MODEL_PATH = "../models/MedGemma-4B-IT"       # <-- SESUAIKAN
M1_MODEL_PATH = "../models/NV-Reason-CXR-3B"     # <-- SESUAIKAN

# Few-shot example pool CSVs
CHEXPERT_PTX_CSV  = "../data/chexpert_ptx_positive.csv"   # <-- SESUAIKAN
CHEXPERT_NORM_CSV = "../data/chexpert_ptx_negative.csv"   # <-- SESUAIKAN

# Test set image directories
CHEXPERT_TEST_NORMAL_DIR = "../data/chexpert_plus/images/internal_test_normal"  # <-- SESUAIKAN
CHEXPERT_TEST_PTX_DIR    = "../data/chexpert_plus/images/internal_test_pneumothorax"  # <-- SESUAIKAN
CHEXPERT_TEST_CSV        = "../data/chexpert_internal_test.csv"  # <-- SESUAIKAN

RSUNAIR_IMAGES_NORMAL = "../data/rs_unair/images/normal"  # <-- SESUAIKAN
RSUNAIR_IMAGES_PTX    = "../data/rs_unair/images/pneumothorax"  # <-- SESUAIKAN
RSUNAIR_LABELS_CSV    = "../data/rs_unair/labels.csv"  # <-- SESUAIKAN

# Few-shot image pool root
POOL_IMAGES_ROOT = "../data/chexpert_plus/images"  # <-- SESUAIKAN

# Experiment config — added k=8
K_SHOTS = [0, 1, 4, 8]
SEEDS = [0, 1, 2, 3, 4]
MAX_NEW_TOKENS = 1024

# Output directories
OUTPUT_DIR_PASS1 = Path("results-sd-pass1-m4-screener")
OUTPUT_DIR_PASS2 = Path("results-sd-pass2-m1-reviewer")
OUTPUT_DIR_PASS1.mkdir(exist_ok=True)
OUTPUT_DIR_PASS2.mkdir(exist_ok=True)

CHECKPOINT_PASS1 = OUTPUT_DIR_PASS1 / "checkpoint_sd_pass1.jsonl"
CHECKPOINT_PASS2 = OUTPUT_DIR_PASS2 / "checkpoint_sd_pass2.jsonl"

print("Config loaded.")
print(f"  M4 model: {M4_MODEL_PATH}")
print(f"  M1 model: {M1_MODEL_PATH}")
print(f"  K-shots: {K_SHOTS}")
print(f"  Seeds: {SEEDS}")
print(f"  Pass 1 output: {OUTPUT_DIR_PASS1}")
print(f"  Pass 2 output: {OUTPUT_DIR_PASS2}")

## 3 — Load Few-Shot Example Selector

In [ ]:
try:
    from example_selector import FewShotSelector
    selector = FewShotSelector(
        positive_csv=CHEXPERT_PTX_CSV,
        negative_csv=CHEXPERT_NORM_CSV,
        images_root=POOL_IMAGES_ROOT,
    )
    USE_SELECTOR = True
    print("FewShotSelector loaded successfully.")
except Exception as e:
    USE_SELECTOR = False
    print(f"FewShotSelector not available: {e}")
    print("Few-shot context will be skipped.")

## 4 — Load Test Sets

In [ ]:
def load_test_images(csv_path, normal_dir, ptx_dir, name):
    """Load test set image paths and labels."""
    df = pd.read_csv(csv_path)
    rows = []
    
    if 'file_id' in df.columns:
        # CheXpert format
        for _, r in df.iterrows():
            fid = r['file_id']
            label_int = 1 if r['label'] == 'pneumothorax' else 0
            img_dir = ptx_dir if label_int == 1 else normal_dir
            img_path = Path(img_dir) / f"{fid}.jpg"
            if not img_path.exists():
                img_path = Path(img_dir) / f"{fid}.png"
            rows.append({'image_id': fid, 'ground_truth': label_int, 'image_path': str(img_path)})
    elif 'image_id' in df.columns:
        # RS UNAIR format
        for _, r in df.iterrows():
            fname = r['image_id']
            if not fname.endswith(('.png', '.jpg')):
                fname = f"{fname}.png"
            label_int = int(r['label'])
            img_dir = ptx_dir if label_int == 1 else normal_dir
            img_path = Path(img_dir) / fname
            rows.append({'image_id': Path(fname).stem, 'ground_truth': label_int, 'image_path': str(img_path)})
    else:
        raise ValueError(f"Unknown CSV format: {df.columns.tolist()}")
    
    result = pd.DataFrame(rows)
    print(f"  {name}: {len(result)} images (PTX={result['ground_truth'].sum()}, Normal={(result['ground_truth']==0).sum()})")
    return result


print("Loading test sets...")
test_sets = [
    {
        'name': 'CheXpert Plus (Internal)',
        'df': load_test_images(CHEXPERT_TEST_CSV, CHEXPERT_TEST_NORMAL_DIR, CHEXPERT_TEST_PTX_DIR, 'CheXpert'),
    },
    {
        'name': 'RS UNAIR (External)',
        'df': load_test_images(RSUNAIR_LABELS_CSV, RSUNAIR_IMAGES_NORMAL, RSUNAIR_IMAGES_PTX, 'RS UNAIR'),
    },
]

imgs_per_round = sum(ts['df'].shape[0] for ts in test_sets)
total_inferences = imgs_per_round * len(K_SHOTS) * len(SEEDS)
print(f"\nImages per (k, seed): {imgs_per_round}")
print(f"Total inferences per pass: {total_inferences}")

---
# ══════════════════════════════════════════════════════════
# PASS 1 — M4 as First Reader (Screener) with Confidence
# ══════════════════════════════════════════════════════════

## 5 — Load M4 Model (MedGemma-4B-IT)

**PENTING:** Gemma 3 harus di-load dengan `bfloat16` compute dtype. float16 menghasilkan pad tokens.

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import Gemma3ForConditionalGeneration

print(f"Loading M4 model: {M4_MODEL_PATH}")
print("Architecture: Gemma 3 (MedGemma-4B-IT)")
print("Quantization: 4-bit NF4 (bfloat16 compute)")

quant_config_gemma = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # CRITICAL: NOT float16!
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

m4_processor = AutoProcessor.from_pretrained(M4_MODEL_PATH)
m4_model = Gemma3ForConditionalGeneration.from_pretrained(
    M4_MODEL_PATH,
    quantization_config=quant_config_gemma,
    device_map="auto",
)
m4_model.eval()

print(f"M4 loaded. Device: {m4_model.device}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 6 — M4 First Reader Prompt & Inference Function

In [ ]:
# ── M4 System Prompt: First Reader / Screener with Confidence ──

SYSTEM_PROMPT_M4_SCREENER = (
    "You are an expert radiologist assistant performing an initial reading "
    "of a chest X-ray (CXR) for pneumothorax screening.\n\n"
    "Analyze the chest X-ray carefully and provide:\n\n"
    "1. CLASSIFICATION: State exactly one of: PNEUMOTHORAX DETECTED or "
    "NO PNEUMOTHORAX DETECTED\n\n"
    "2. CONFIDENCE: Your confidence level as a percentage (0-100%).\n"
    "   - 90-100%: Very confident, clear and unambiguous findings\n"
    "   - 70-89%: Confident, findings are present but may be subtle\n"
    "   - 50-69%: Moderate, some suggestive signs but uncertain\n"
    "   - 30-49%: Low confidence, ambiguous findings\n"
    "   - 0-29%: Very low confidence, minimal evidence\n\n"
    "3. KEY EVIDENCE: Describe the specific radiographic findings that "
    "support your classification in 1-2 sentences. Mention specific "
    "anatomical locations and signs observed.\n\n"
    "Important:\n"
    "- Base your classification on radiographic evidence visible in the image.\n"
    "- If you see signs consistent with pneumothorax (even subtle), classify "
    "as PNEUMOTHORAX DETECTED.\n"
    "- If the lung fields are clear with no evidence of pneumothorax, classify "
    "as NO PNEUMOTHORAX DETECTED.\n"
    "- Be honest about your confidence — low confidence is acceptable and "
    "helps the reviewing specialist focus their attention."
)

USER_QUERY_M4_SCREENER = (
    "Analyze this chest X-ray for pneumothorax. "
    "Provide your classification, confidence level (0-100%), and key evidence."
)


def generate_m4_screening(
    image_path: str,
    few_shot_context: str = "",
    max_new_tokens: int = 1024,
) -> tuple:
    """
    Run M4 as first reader (screener) with confidence output.
    Returns: (generated_text, inference_time_seconds)
    """
    img = Image.open(image_path)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    content_parts = []
    if few_shot_context:
        content_parts.append({"type": "text", "text": few_shot_context})
    content_parts.append({"type": "image", "image": img})
    content_parts.append({"type": "text", "text": USER_QUERY_M4_SCREENER})
    
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT_M4_SCREENER}]},
        {"role": "user", "content": content_parts},
    ]
    
    text = m4_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = m4_processor(text=text, images=[img], return_tensors="pt").to(m4_model.device)
    input_len = inputs["input_ids"].shape[1]
    
    t0 = time.time()
    with torch.no_grad():
        output_ids = m4_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    elapsed = time.time() - t0
    
    generated_ids = output_ids[:, input_len:]
    result = m4_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    del inputs, output_ids, generated_ids
    torch.cuda.empty_cache()
    
    return result, elapsed


print("generate_m4_screening() defined.")
print(f"System prompt length: {len(SYSTEM_PROMPT_M4_SCREENER)} chars")

## 6b — Quick Test Pass 1 (1 image)

In [ ]:
# Test on 1 image to verify output format
test_row = test_sets[0]['df'].iloc[0]
print(f"Test image: {test_row['image_id']} (GT={test_row['ground_truth']})")

test_result, test_time = generate_m4_screening(test_row['image_path'])
print(f"\nM4 output ({test_time:.1f}s):")
print(test_result)
print("\n" + "="*60)
print("Verify: output should contain CLASSIFICATION, CONFIDENCE (%), and KEY EVIDENCE")
has_class = 'PNEUMOTHORAX' in test_result.upper()
has_conf = '%' in test_result or 'CONFIDENCE' in test_result.upper()
has_evidence = 'EVIDENCE' in test_result.upper() or 'evidence' in test_result.lower()
print(f"  Has classification: {has_class}")
print(f"  Has confidence: {has_conf}")
print(f"  Has evidence: {has_evidence}")

## 7 — Pass 1 Main Inference Loop (M4 Screener)

In [ ]:
# Load checkpoint
done_keys_p1, _ = load_checkpoint(CHECKPOINT_PASS1)
print(f"Pass 1 checkpoint: {len(done_keys_p1)} records already done.")

remaining = total_inferences - len(done_keys_p1)
print(f"Remaining: {remaining} inferences")

In [ ]:
# ═══════════════════════════════════════════════════════════
# PASS 1 MAIN LOOP — M4 First Reader (Screener)
# ═══════════════════════════════════════════════════════════

batch_buffer = []
BATCH_SIZE = 10
errors_p1 = []

pbar = tqdm(total=remaining, desc="Pass 1: M4 Screener")

for k in K_SHOTS:
    for seed in SEEDS:
        # Build few-shot context
        few_shot_context = ""
        if k > 0 and USE_SELECTOR:
            try:
                examples = selector.select(strategy="random", k=k, seed=seed)
                few_shot_context = build_few_shot_context_v2(examples, k)
            except Exception as e:
                print(f"Warning: few-shot context failed k={k} seed={seed}: {e}")
        
        for ts in test_sets:
            ts_name = ts['name']
            
            for _, img_row in ts['df'].iterrows():
                image_id = img_row['image_id']
                key = (k, seed, ts_name, image_id)
                
                if key in done_keys_p1:
                    continue
                
                try:
                    m4_text, m4_time = generate_m4_screening(
                        img_row['image_path'],
                        few_shot_context=few_shot_context,
                        max_new_tokens=MAX_NEW_TOKENS,
                    )
                    
                    record = {
                        'experiment_type': 'sd_pass1_m4_screener',
                        'k_shot': k,
                        'strategy': 'random',
                        'seed': seed,
                        'test_set': ts_name,
                        'image_id': image_id,
                        'ground_truth': int(img_row['ground_truth']),
                        'm4_generated_text': m4_text,
                        'm4_inference_time': round(m4_time, 3),
                        'timestamp': datetime.now().isoformat(),
                    }
                    
                    batch_buffer.append(record)
                    done_keys_p1.add(key)
                    
                    if len(batch_buffer) >= BATCH_SIZE:
                        append_jsonl(CHECKPOINT_PASS1, batch_buffer)
                        batch_buffer = []
                
                except Exception as e:
                    errors_p1.append(f"k={k} seed={seed} {ts_name} {image_id}: {e}")
                
                pbar.update(1)

# Flush remaining
if batch_buffer:
    append_jsonl(CHECKPOINT_PASS1, batch_buffer)
    batch_buffer = []

pbar.close()
print(f"\nPass 1 complete. Total done: {len(done_keys_p1)}")
print(f"Errors: {len(errors_p1)}")
if errors_p1:
    for e in errors_p1[:5]:
        print(f"  {e}")

## 8 — Unload M4, Free VRAM

In [ ]:
print(f"VRAM before unload: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

del m4_model
del m4_processor
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after unload: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("M4 unloaded. Ready for M1.")

---
# ══════════════════════════════════════════════════════════
# PASS 2 — M1 as Specialist Reviewer with Confidence
# ══════════════════════════════════════════════════════════

## 9 — Load Pass 1 Results (M4 outputs)

In [ ]:
# Load M4 Pass 1 results
p1_records = []
with open(CHECKPOINT_PASS1, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            p1_records.append(json.loads(line))

df_pass1 = pd.DataFrame(p1_records)
print(f"Pass 1 records loaded: {len(df_pass1)}")
print(f"  k_shots: {sorted(df_pass1['k_shot'].unique())}")
print(f"  seeds: {sorted(df_pass1['seed'].unique())}")
print(f"  test_sets: {df_pass1['test_set'].unique().tolist()}")


def get_m4_output(k_shot, seed, test_set, image_id):
    """Get M4's generated text for a specific (k, seed, test_set, image_id)."""
    mask = (
        (df_pass1['k_shot'] == k_shot)
        & (df_pass1['seed'] == seed)
        & (df_pass1['test_set'] == test_set)
        & (df_pass1['image_id'] == image_id)
    )
    matches = df_pass1[mask]
    if len(matches) == 0:
        return None
    return matches.iloc[0]['m4_generated_text']

## 10 — Load M1 Model (NV-Reason-CXR-3B, Qwen2.5-VL)

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

print(f"Loading M1 model: {M1_MODEL_PATH}")
print("Architecture: Qwen2.5-VL (NV-Reason-CXR-3B)")
print("Quantization: 4-bit NF4")

quant_config_qwen = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

m1_processor = AutoProcessor.from_pretrained(
    M1_MODEL_PATH,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)

m1_model = AutoModelForImageTextToText.from_pretrained(
    M1_MODEL_PATH,
    quantization_config=quant_config_qwen,
    device_map="auto",
)
m1_model.eval()

print(f"M1 loaded. Device: {m1_model.device}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 11 — M1 Specialist Reviewer Prompt & Inference Function

In [ ]:
# ── M1 System Prompt: Specialist Reviewer with Confidence ──

SYSTEM_PROMPT_M1_REVIEWER = (
    "You are a specialist radiologist performing a second reading of a chest "
    "X-ray (CXR) that has been initially screened by a general AI reader.\n\n"
    "The first reader's assessment is provided below, including their "
    "classification, confidence level, and key evidence.\n\n"
    "Your role as specialist reviewer:\n"
    "1. EVALUATE the first reader's assessment — do their findings match "
    "the radiographic evidence?\n"
    "2. PAY SPECIAL ATTENTION to areas of low confidence or ambiguous "
    "findings reported by the first reader\n"
    "3. PERFORM YOUR OWN thorough examination — look for subtle signs "
    "the first reader may have missed, especially:\n"
    "   - Visceral pleural lines (particularly at the apex)\n"
    "   - Areas of hyperlucency\n"
    "   - Absent lung markings\n"
    "   - Deep sulcus sign (on supine films)\n"
    "4. Provide your final assessment in this format:\n\n"
    "FIRST READER EVALUATION: [2-3 sentences evaluating the first reader's "
    "claims and confidence level]\n"
    "MY OBSERVATIONS: [2-3 sentences describing what you observe, especially "
    "any findings the first reader missed or misinterpreted]\n"
    "CLASSIFICATION: [PNEUMOTHORAX DETECTED or NO PNEUMOTHORAX DETECTED]\n"
    "CONFIDENCE: [0-100%]\n"
    "KEY EVIDENCE: [1-2 sentences with specific anatomical findings]\n\n"
    "Important:\n"
    "- You are the specialist — your assessment takes priority\n"
    "- If the first reader missed subtle findings, identify them\n"
    "- If the first reader over-interpreted normal variants, correct them\n"
    "- Be thorough but evidence-based in your classification"
)

USER_QUERY_M1_REVIEWER = (
    "Based on the first reader's assessment above and your own specialist "
    "examination of this chest X-ray, provide your second reading."
)


def build_m1_reviewer_input(m4_output: str, few_shot_context: str = "") -> str:
    """Build user text content for M1 reviewer prompt."""
    parts = []
    if few_shot_context:
        parts.append(few_shot_context)
        parts.append("")
    parts.append("FIRST READER'S ASSESSMENT:")
    parts.append(m4_output.strip())
    return "\n".join(parts)


def generate_m1_review(
    image_path: str,
    m4_output: str,
    few_shot_context: str = "",
    max_new_tokens: int = 1024,
) -> tuple:
    """
    Run M1 as specialist reviewer with M4's output as context.
    Returns: (generated_text, inference_time_seconds)
    """
    img = Image.open(image_path)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    user_text = build_m1_reviewer_input(m4_output, few_shot_context)
    
    content_parts = []
    if user_text:
        content_parts.append({"type": "text", "text": user_text})
    content_parts.append({"type": "image", "image": img})
    content_parts.append({"type": "text", "text": USER_QUERY_M1_REVIEWER})
    
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT_M1_REVIEWER}]},
        {"role": "user", "content": content_parts},
    ]
    
    text = m1_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = m1_processor(text=text, images=[img], return_tensors="pt").to(m1_model.device)
    input_len = inputs["input_ids"].shape[1]
    
    t0 = time.time()
    with torch.no_grad():
        output_ids = m1_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    elapsed = time.time() - t0
    
    generated_ids = output_ids[:, input_len:]
    result = m1_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    del inputs, output_ids, generated_ids
    torch.cuda.empty_cache()
    
    return result, elapsed


print("generate_m1_review() defined.")
print(f"System prompt length: {len(SYSTEM_PROMPT_M1_REVIEWER)} chars")

## 11b — Quick Test Pass 2 (1 image)

In [ ]:
# Test on same image as Pass 1
test_row = test_sets[0]['df'].iloc[0]
test_m4_output = get_m4_output(0, 0, test_sets[0]['name'], test_row['image_id'])

print(f"Test image: {test_row['image_id']} (GT={test_row['ground_truth']})")
print(f"\nM4 said: {test_m4_output[:200]}...")
print("\n" + "="*60)

test_m1_result, test_m1_time = generate_m1_review(test_row['image_path'], test_m4_output)
print(f"\nM1 reviewer output ({test_m1_time:.1f}s):")
print(test_m1_result)
print("\n" + "="*60)
print("Verify: output should contain FIRST READER EVALUATION, MY OBSERVATIONS,")
print("        CLASSIFICATION, CONFIDENCE (%), KEY EVIDENCE")

## 12 — Pass 2 Main Inference Loop (M1 Reviewer)

In [ ]:
# Load checkpoint
done_keys_p2, _ = load_checkpoint(CHECKPOINT_PASS2)
print(f"Pass 2 checkpoint: {len(done_keys_p2)} records already done.")

remaining_p2 = total_inferences - len(done_keys_p2)
print(f"Remaining: {remaining_p2} inferences")

In [ ]:
# ═══════════════════════════════════════════════════════════
# PASS 2 MAIN LOOP — M1 Specialist Reviewer
# ═══════════════════════════════════════════════════════════

batch_buffer = []
BATCH_SIZE = 10
errors_p2 = []

pbar = tqdm(total=remaining_p2, desc="Pass 2: M1 Reviewer")

for k in K_SHOTS:
    for seed in SEEDS:
        # Build few-shot context (same as Pass 1)
        few_shot_context = ""
        if k > 0 and USE_SELECTOR:
            try:
                examples = selector.select(strategy="random", k=k, seed=seed)
                few_shot_context = build_few_shot_context_v2(examples, k)
            except Exception as e:
                print(f"Warning: few-shot context failed k={k} seed={seed}: {e}")
        
        for ts in test_sets:
            ts_name = ts['name']
            
            for _, img_row in ts['df'].iterrows():
                image_id = img_row['image_id']
                key = (k, seed, ts_name, image_id)
                
                if key in done_keys_p2:
                    continue
                
                # Get M4's Pass 1 output
                m4_output = get_m4_output(k, seed, ts_name, image_id)
                if m4_output is None:
                    errors_p2.append(f"M4 output missing: k={k} seed={seed} {ts_name} {image_id}")
                    continue
                
                try:
                    m1_text, m1_time = generate_m1_review(
                        img_row['image_path'],
                        m4_output,
                        few_shot_context=few_shot_context,
                        max_new_tokens=MAX_NEW_TOKENS,
                    )
                    
                    record = {
                        'experiment_type': 'sd_pass2_m1_reviewer',
                        'k_shot': k,
                        'strategy': 'random',
                        'seed': seed,
                        'test_set': ts_name,
                        'image_id': image_id,
                        'ground_truth': int(img_row['ground_truth']),
                        'm4_generated_text': m4_output,
                        'm1_generated_text': m1_text,
                        'm1_inference_time': round(m1_time, 3),
                        'timestamp': datetime.now().isoformat(),
                    }
                    
                    batch_buffer.append(record)
                    done_keys_p2.add(key)
                    
                    if len(batch_buffer) >= BATCH_SIZE:
                        append_jsonl(CHECKPOINT_PASS2, batch_buffer)
                        batch_buffer = []
                
                except Exception as e:
                    errors_p2.append(f"k={k} seed={seed} {ts_name} {image_id}: {e}")
                
                pbar.update(1)

# Flush remaining
if batch_buffer:
    append_jsonl(CHECKPOINT_PASS2, batch_buffer)
    batch_buffer = []

pbar.close()
print(f"\nPass 2 complete. Total done: {len(done_keys_p2)}")
print(f"Errors: {len(errors_p2)}")
if errors_p2:
    for e in errors_p2[:5]:
        print(f"  {e}")

---
# ══════════════════════════════════════════════════════════
# ANALYSIS
# ══════════════════════════════════════════════════════════

## 13 — Consolidate Results & Extract Predictions

In [ ]:
# Load all Pass 2 results (contains both M4 and M1 outputs)
records = []
with open(CHECKPOINT_PASS2, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df_all = pd.DataFrame(records)
print(f"Total records: {len(df_all)}")

# Save consolidated
all_results_path = OUTPUT_DIR_PASS2 / "all_results_sd_m4_m1.jsonl"
with open(all_results_path, 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f"Saved to {all_results_path}")

In [ ]:
def extract_confidence(text):
    """Extract confidence percentage from generated text."""
    # Try patterns: "CONFIDENCE: 73%", "Confidence: 73", "73%"
    patterns = [
        r'CONFIDENCE:\s*(\d+)\s*%',
        r'CONFIDENCE:\s*(\d+)',
        r'confidence[:\s]+(\d+)\s*%',
        r'(\d+)\s*%\s*confident',
        r'(\d+)%',
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            val = int(match.group(1))
            if 0 <= val <= 100:
                return val
    return None


def extract_classification(text):
    """Extract binary classification from text."""
    upper = text.upper()
    # Check for explicit classification line first
    class_match = re.search(r'CLASSIFICATION:\s*(.+)', upper)
    if class_match:
        line = class_match.group(1).strip()
        if 'NO PNEUMOTHORAX' in line:
            return 0
        elif 'PNEUMOTHORAX' in line:
            return 1
    # Fallback: check last occurrence
    if 'NO PNEUMOTHORAX' in upper:
        return 0
    elif 'PNEUMOTHORAX DETECTED' in upper:
        return 1
    return None


# Extract predictions and confidence for both M4 and M1
df_all['m4_pred'] = df_all['m4_generated_text'].apply(extract_classification)
df_all['m4_confidence'] = df_all['m4_generated_text'].apply(extract_confidence)
df_all['m1_pred'] = df_all['m1_generated_text'].apply(extract_classification)
df_all['m1_confidence'] = df_all['m1_generated_text'].apply(extract_confidence)

# Summary
print("Extraction summary:")
print(f"  M4 pred: {df_all['m4_pred'].notna().sum()}/{len(df_all)} extracted")
print(f"  M4 conf: {df_all['m4_confidence'].notna().sum()}/{len(df_all)} extracted")
print(f"  M1 pred: {df_all['m1_pred'].notna().sum()}/{len(df_all)} extracted")
print(f"  M1 conf: {df_all['m1_confidence'].notna().sum()}/{len(df_all)} extracted")
print()
print("M4 confidence distribution:")
print(df_all['m4_confidence'].describe())
print()
print("M1 confidence distribution:")
print(df_all['m1_confidence'].describe())

## 14 — Aggregation Strategies D1-D4

In [ ]:
def aggregate_d1(m4_pred, m4_conf, m1_pred, m1_conf):
    """D1: M1 reviewer decision (baseline)."""
    return m1_pred


def aggregate_d2(m4_pred, m4_conf, m1_pred, m1_conf):
    """D2: Confidence-weighted vote."""
    c4 = m4_conf if m4_conf is not None else 50
    c1 = m1_conf if m1_conf is not None else 50
    p4 = m4_pred if m4_pred is not None else 0
    p1 = m1_pred if m1_pred is not None else 0
    weighted = (c4 * p4 + c1 * p1) / (c4 + c1 + 1e-6)
    return 1 if weighted >= 0.5 else 0


def aggregate_d3(m4_pred, m4_conf, m1_pred, m1_conf, tau_high=80, tau_low=40):
    """D3: Confidence-threshold cascade."""
    c4 = m4_conf if m4_conf is not None else 50
    c1 = m1_conf if m1_conf is not None else 50
    p4 = m4_pred if m4_pred is not None else 0
    p1 = m1_pred if m1_pred is not None else 0
    if c4 >= tau_high:
        return p4
    elif c4 <= tau_low:
        return p1
    else:
        return p1 if c1 > c4 else p4


def aggregate_d4(m4_pred, m4_conf, m1_pred, m1_conf):
    """D4: Agreement-aware."""
    c4 = m4_conf if m4_conf is not None else 50
    c1 = m1_conf if m1_conf is not None else 50
    p4 = m4_pred if m4_pred is not None else 0
    p1 = m1_pred if m1_pred is not None else 0
    if p4 == p1:
        return p4  # agree
    else:
        return p1 if c1 >= c4 else p4  # disagree → trust higher confidence


# Apply all strategies
df_all['pred_d1'] = df_all.apply(lambda r: aggregate_d1(r['m4_pred'], r['m4_confidence'], r['m1_pred'], r['m1_confidence']), axis=1)
df_all['pred_d2'] = df_all.apply(lambda r: aggregate_d2(r['m4_pred'], r['m4_confidence'], r['m1_pred'], r['m1_confidence']), axis=1)
df_all['pred_d3'] = df_all.apply(lambda r: aggregate_d3(r['m4_pred'], r['m4_confidence'], r['m1_pred'], r['m1_confidence']), axis=1)
df_all['pred_d4'] = df_all.apply(lambda r: aggregate_d4(r['m4_pred'], r['m4_confidence'], r['m1_pred'], r['m1_confidence']), axis=1)

print("Aggregation strategies applied: D1, D2, D3, D4")
for d in ['pred_d1', 'pred_d2', 'pred_d3', 'pred_d4']:
    pos = (df_all[d] == 1).sum()
    print(f"  {d}: {pos} positive / {len(df_all)} total ({100*pos/len(df_all):.1f}%)")

## 15 — Compute Metrics per Strategy

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, accuracy_score

results_per_seed = []

strategies = {
    'D1_M1_Decision': 'pred_d1',
    'D2_Conf_Weighted': 'pred_d2',
    'D3_Threshold_Cascade': 'pred_d3',
    'D4_Agreement_Aware': 'pred_d4',
    'M4_Screener_Only': 'm4_pred',
}

for k in K_SHOTS:
    for seed in SEEDS:
        for ts in test_sets:
            ts_name = ts['name']
            mask = (df_all['k_shot'] == k) & (df_all['seed'] == seed) & (df_all['test_set'] == ts_name)
            subset = df_all[mask]
            
            if len(subset) == 0:
                continue
            
            gt = subset['ground_truth'].values
            n_pos = gt.sum()
            n_neg = (gt == 0).sum()
            
            for strat_name, col in strategies.items():
                preds = subset[col].fillna(0).astype(int).values
                
                tp = ((preds == 1) & (gt == 1)).sum()
                tn = ((preds == 0) & (gt == 0)).sum()
                fp = ((preds == 1) & (gt == 0)).sum()
                fn = ((preds == 0) & (gt == 1)).sum()
                
                sens = tp / (tp + fn) if (tp + fn) > 0 else 0
                spec = tn / (tn + fp) if (tn + fp) > 0 else 0
                prec = tp / (tp + fp) if (tp + fp) > 0 else 0
                f1 = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0
                acc = (tp + tn) / len(preds)
                pbr = (tp + fp) / n_pos if n_pos > 0 else 0
                
                results_per_seed.append({
                    'method': strat_name,
                    'k_shot': k,
                    'seed': seed,
                    'test_set': ts_name,
                    'sensitivity': round(sens, 4),
                    'specificity': round(spec, 4),
                    'precision': round(prec, 4),
                    'f1': round(f1, 4),
                    'accuracy': round(acc, 4),
                    'pbr': round(pbr, 4),
                    'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
                })

df_per_seed = pd.DataFrame(results_per_seed)
df_per_seed.to_csv(OUTPUT_DIR_PASS2 / 'sd_per_seed.csv', index=False)
print(f"Per-seed results: {len(df_per_seed)} rows")

# Summary: mean +/- std across seeds
metrics_cols = ['sensitivity', 'specificity', 'precision', 'f1', 'accuracy', 'pbr']
summary = df_per_seed.groupby(['method', 'k_shot', 'test_set'])[metrics_cols].agg(['mean', 'std']).reset_index()
summary.columns = ['_'.join(col).rstrip('_') for col in summary.columns]
summary.to_csv(OUTPUT_DIR_PASS2 / 'sd_summary.csv', index=False)
print(f"Summary saved: {OUTPUT_DIR_PASS2 / 'sd_summary.csv'}")

## 16 — Display Results Table

In [ ]:
# Show summary table
for ts_name in ['CheXpert Plus (Internal)', 'RS UNAIR (External)']:
    print(f"\n{'='*80}")
    print(f"  {ts_name}")
    print(f"{'='*80}")
    print(f"{'Method':<25} {'k':>3} {'Sens':>8} {'Spec':>8} {'Prec':>8} {'F1':>8} {'PBR':>8}")
    print('-' * 80)
    
    for k in K_SHOTS:
        for strat_name in strategies.keys():
            mask = (df_per_seed['method'] == strat_name) & (df_per_seed['k_shot'] == k) & (df_per_seed['test_set'] == ts_name)
            sub = df_per_seed[mask]
            if len(sub) == 0:
                continue
            print(f"{strat_name:<25} {k:>3} {sub['sensitivity'].mean():>8.3f} {sub['specificity'].mean():>8.3f} {sub['precision'].mean():>8.3f} {sub['f1'].mean():>8.3f} {sub['pbr'].mean():>8.3f}")
        print()

## 17 — Confidence Analysis

In [ ]:
import matplotlib.pyplot as plt

FIGURES_DIR = OUTPUT_DIR_PASS2.parent / 'figures-sd-m4-m1'
FIGURES_DIR.mkdir(exist_ok=True)

# Confidence distribution: M4 vs M1, correct vs incorrect
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Confidence Distribution: M4 Screener vs M1 Reviewer', fontsize=14)

for idx, (model_name, conf_col, pred_col) in enumerate([
    ('M4 Screener', 'm4_confidence', 'm4_pred'),
    ('M1 Reviewer', 'm1_confidence', 'm1_pred'),
]):
    valid = df_all[df_all[conf_col].notna()].copy()
    valid['correct'] = (valid[pred_col] == valid['ground_truth']).astype(int)
    
    ax_correct = axes[idx, 0]
    ax_correct.hist(valid[valid['correct']==1][conf_col], bins=20, alpha=0.7, label='Correct', color='green')
    ax_correct.hist(valid[valid['correct']==0][conf_col], bins=20, alpha=0.7, label='Incorrect', color='red')
    ax_correct.set_title(f'{model_name} — Correct vs Incorrect')
    ax_correct.set_xlabel('Confidence (%)')
    ax_correct.legend()
    
    ax_class = axes[idx, 1]
    ptx = valid[valid[pred_col]==1][conf_col]
    no_ptx = valid[valid[pred_col]==0][conf_col]
    ax_class.hist(ptx, bins=20, alpha=0.7, label='PTX Detected', color='orange')
    ax_class.hist(no_ptx, bins=20, alpha=0.7, label='No PTX', color='blue')
    ax_class.set_title(f'{model_name} — By Classification')
    ax_class.set_xlabel('Confidence (%)')
    ax_class.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig_confidence_distribution.pdf', bbox_inches='tight')
plt.show()
print("Saved confidence distribution figure.")

In [ ]:
# Calibration curve (reliability diagram)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Calibration: Confidence vs Actual Accuracy', fontsize=14)

for idx, (model_name, conf_col, pred_col) in enumerate([
    ('M4 Screener', 'm4_confidence', 'm4_pred'),
    ('M1 Reviewer', 'm1_confidence', 'm1_pred'),
]):
    valid = df_all[df_all[conf_col].notna()].copy()
    valid['correct'] = (valid[pred_col] == valid['ground_truth']).astype(int)
    
    # Bin by confidence
    bins = list(range(0, 101, 10))
    valid['conf_bin'] = pd.cut(valid[conf_col], bins=bins, labels=[f'{b}-{b+10}' for b in bins[:-1]])
    
    cal = valid.groupby('conf_bin').agg(
        mean_conf=(conf_col, 'mean'),
        mean_acc=('correct', 'mean'),
        count=('correct', 'count'),
    ).dropna()
    
    ax = axes[idx]
    ax.plot([0, 100], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    ax.bar(cal['mean_conf'], cal['mean_acc'], width=8, alpha=0.7, color='steelblue')
    ax.set_xlabel('Mean Confidence (%)')
    ax.set_ylabel('Actual Accuracy')
    ax.set_title(f'{model_name}')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 1.05)
    ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_calibration.png', dpi=150, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig_calibration.pdf', bbox_inches='tight')
plt.show()
print("Saved calibration figure.")

## 18 — F1 Comparison Across Strategies

In [ ]:
# F1 plot: all D strategies vs M4-only across k-shots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('F1 Score: S-D Aggregation Strategies', fontsize=14)

colors = {'D1_M1_Decision': 'red', 'D2_Conf_Weighted': 'blue', 
          'D3_Threshold_Cascade': 'green', 'D4_Agreement_Aware': 'purple',
          'M4_Screener_Only': 'gray'}

for idx, ts_name in enumerate(['CheXpert Plus (Internal)', 'RS UNAIR (External)']):
    ax = axes[idx]
    ts_short = 'Internal' if 'Internal' in ts_name else 'External'
    
    for strat_name in strategies.keys():
        f1_means = []
        f1_stds = []
        for k in K_SHOTS:
            mask = (df_per_seed['method'] == strat_name) & (df_per_seed['k_shot'] == k) & (df_per_seed['test_set'] == ts_name)
            sub = df_per_seed[mask]
            f1_means.append(sub['f1'].mean())
            f1_stds.append(sub['f1'].std())
        
        ax.errorbar(K_SHOTS, f1_means, yerr=f1_stds, marker='o', label=strat_name,
                   color=colors.get(strat_name, 'black'), capsize=3)
    
    ax.set_xlabel('k per class')
    ax.set_ylabel('F1 Score')
    ax.set_title(ts_short)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_f1_strategies.png', dpi=150, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig_f1_strategies.pdf', bbox_inches='tight')
plt.show()

## 19 — Qualitative Examples

In [ ]:
# Show interesting cases
k4_s0 = df_all[(df_all['k_shot'] == 4) & (df_all['seed'] == 0)]

# Case 1: M4 misses, M1 catches (the ideal S-D behavior)
print("=== CASE 1: M4 says NO but M1 catches PTX (ideal S-D) ===")
catches = k4_s0[(k4_s0['ground_truth'] == 1) & (k4_s0['m4_pred'] == 0) & (k4_s0['m1_pred'] == 1)]
if len(catches) > 0:
    r = catches.iloc[0]
    print(f"Image: {r['image_id']} | GT: PTX")
    print(f"\nM4: {r['m4_generated_text'][:300]}")
    print(f"\nM1: {r['m1_generated_text'][:400]}")
else:
    print("(No cases found)")

print("\n" + "="*60)

# Case 2: Both agree correctly
print("\n=== CASE 2: Both agree — PTX detected (concordant TP) ===")
agree_tp = k4_s0[(k4_s0['ground_truth'] == 1) & (k4_s0['m4_pred'] == 1) & (k4_s0['m1_pred'] == 1)]
if len(agree_tp) > 0:
    r = agree_tp.iloc[0]
    print(f"Image: {r['image_id']} | GT: PTX")
    print(f"M4 confidence: {r['m4_confidence']}% | M1 confidence: {r['m1_confidence']}%")
    print(f"\nM4: {r['m4_generated_text'][:200]}")
    print(f"\nM1: {r['m1_generated_text'][:200]}")
else:
    print("(No cases found)")

print("\n" + "="*60)

# Case 3: M4 detects, M1 correctly overrides to NO (FP correction)
print("\n=== CASE 3: M4 says PTX (FP), M1 corrects to NO ===")
corrections = k4_s0[(k4_s0['ground_truth'] == 0) & (k4_s0['m4_pred'] == 1) & (k4_s0['m1_pred'] == 0)]
if len(corrections) > 0:
    r = corrections.iloc[0]
    print(f"Image: {r['image_id']} | GT: Normal")
    print(f"M4 confidence: {r['m4_confidence']}% | M1 confidence: {r['m1_confidence']}%")
    print(f"\nM4: {r['m4_generated_text'][:200]}")
    print(f"\nM1: {r['m1_generated_text'][:300]}")
else:
    print("(No cases found)")

# Summary counts
print("\n" + "="*60)
print("\nAgreement/Disagreement at k=4, seed=0:")
agree = (k4_s0['m4_pred'] == k4_s0['m1_pred']).sum()
disagree = (k4_s0['m4_pred'] != k4_s0['m1_pred']).sum()
print(f"  Agree: {agree} ({100*agree/len(k4_s0):.1f}%)")
print(f"  Disagree: {disagree} ({100*disagree/len(k4_s0):.1f}%)")
# When disagree, who is right?
disagree_mask = k4_s0['m4_pred'] != k4_s0['m1_pred']
disagree_cases = k4_s0[disagree_mask]
m1_right = (disagree_cases['m1_pred'] == disagree_cases['ground_truth']).sum()
m4_right = (disagree_cases['m4_pred'] == disagree_cases['ground_truth']).sum()
print(f"  When disagree — M1 correct: {m1_right}, M4 correct: {m4_right}")

## 20 — Key Findings

In [ ]:
print('=' * 80)
print('S-D REVERSED DURING-INFERENCE: M4→M1 CONFIDENCE-AWARE — KEY FINDINGS')
print('=' * 80)

# Best strategy per k
for ts_name in ['CheXpert Plus (Internal)', 'RS UNAIR (External)']:
    ts_short = 'Internal' if 'Internal' in ts_name else 'External'
    print(f"\n{ts_short}:")
    for k in K_SHOTS:
        mask = (df_per_seed['k_shot'] == k) & (df_per_seed['test_set'] == ts_name)
        sub = df_per_seed[mask]
        best = sub.groupby('method')['f1'].mean().sort_values(ascending=False)
        print(f"  k={k}: Best = {best.index[0]} (F1={best.iloc[0]:.3f})")

# Confidence stats
print(f"\nConfidence extraction rate:")
print(f"  M4: {df_all['m4_confidence'].notna().mean()*100:.1f}%")
print(f"  M1: {df_all['m1_confidence'].notna().mean()*100:.1f}%")
print(f"\nMean confidence:")
print(f"  M4: {df_all['m4_confidence'].mean():.1f}%")
print(f"  M1: {df_all['m1_confidence'].mean():.1f}%")

## 21 — Export

In [ ]:
print('EXPORTED FILES:')
print(f'  Pass 1 checkpoint:  {CHECKPOINT_PASS1}')
print(f'  Pass 2 checkpoint:  {CHECKPOINT_PASS2}')
print(f'  Consolidated:       {OUTPUT_DIR_PASS2 / "all_results_sd_m4_m1.jsonl"}')
print(f'  Per-seed metrics:   {OUTPUT_DIR_PASS2 / "sd_per_seed.csv"}')
print(f'  Summary metrics:    {OUTPUT_DIR_PASS2 / "sd_summary.csv"}')
print(f'  Figures:            {FIGURES_DIR}/')